# 01 – EDA PKLot (dronefreak/PKLot, định dạng YOLO)

Dataset: https://huggingface.co/datasets/dronefreak/PKLot – 12.416 ảnh 1280×720 từ 3 camera (PUCPR, UFPR04, UFPR05),
mỗi ô đỗ là 1 bbox với lớp `0 = vacant`, `1 = occupied`. Train/valid/test đã được chia sẵn theo **(camera, ngày)**.

Notebook này:
1. Tải / tìm dataset, đọc toàn bộ label thành bảng.
2. Thống kê theo tập, camera, lớp; tỷ lệ lấp đầy theo giờ; kích thước bbox.
3. Kiểm tra không có (camera, ngày) nào xuất hiện ở nhiều tập.
4. Vẽ bbox lên ảnh mẫu của từng camera.

**Trên Kaggle:** bật *Settings → Internet* (lần đầu tải ~3.9 GB), hoặc gắn sẵn dataset qua *Add Input*.

In [ ]:
import json
import os
from pathlib import Path

HF_REPO = "dronefreak/PKLot"
SPLITS = ["train", "valid", "test"]
IMG_W, IMG_H = 1280, 720
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path(".")
DOWNLOAD_DIR = Path("/tmp/pklot")   # không lưu vào output, chỉ dùng trong phiên

def get_hf_token():
    # Có token thì giới hạn tải tính theo tài khoản, không bị chung IP với người khác trên Kaggle (lỗi 429)
    try:
        from kaggle_secrets import UserSecretsClient
    except ImportError:   # chạy ngoài Kaggle
        return os.environ.get("HF_TOKEN")
    try:
        return UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        print("Không đọc được secret HF_TOKEN:", repr(e))
        return os.environ.get("HF_TOKEN")

def write_yolo_labels(data_root: Path):
    # Tạo labels/*.txt từ metadata.jsonl (bbox COCO pixel → YOLO chuẩn hóa), khớp 100% file label gốc
    for split in SPLITS:
        out = data_root / "labels" / split
        out.mkdir(parents=True, exist_ok=True)
        for line in (data_root / "images" / split / "metadata.jsonl").read_text().splitlines():
            rec = json.loads(line)
            rows = [f"{c} {(x + w / 2) / IMG_W:.6f} {(y + h / 2) / IMG_H:.6f} {w / IMG_W:.6f} {h / IMG_H:.6f}"
                    for (x, y, w, h), c in zip(rec["objects"]["bbox"], rec["objects"]["categories"])]
            (out / f"{Path(rec['file_name']).stem}.txt").write_text("\n".join(rows) + "\n")

def download_pklot(dest: Path) -> Path:
    from huggingface_hub import snapshot_download   # cần bật Internet trong Settings
    token = get_hf_token()
    if not token:
        # Dừng ngay thay vì tải ẩn danh hàng giờ vì bị giới hạn (HTTP 429)
        raise RuntimeError(
            "Chưa có HF_TOKEN. Trên Kaggle: Add-ons → Secrets → thêm secret tên đúng 'HF_TOKEN' "
            "và ĐÁNH DẤU ô bên cạnh để gắn vào notebook này, rồi chạy lại. "
            "Hoặc Add Input dataset PKLot đã lưu để khỏi phải tải.")
    # Chỉ tải ảnh + metadata.jsonl; label tự tạo lại → giảm một nửa số request.
    # Chạy lại cell sẽ bỏ qua các file đã tải xong.
    snapshot_download(HF_REPO, repo_type="dataset", local_dir=dest, token=token, max_workers=8,
                      allow_patterns=["data/data.yaml", "data/images/*"])
    write_yolo_labels(dest / "data")
    (dest / "data" / ".complete").touch()
    return dest / "data"

def find_data_root() -> Path:
    # 1) Dataset đã gắn qua Add Input  2) đã tải xong trước đó  3) tải từ Hugging Face
    for depth in range(1, 6):   # không dùng "**" để tránh duyệt hết hàng chục nghìn ảnh
        for yaml_path in Path("/kaggle/input").glob("*/" * depth + "data.yaml"):
            if (yaml_path.parent / "labels").is_dir():
                return yaml_path.parent
    if (DOWNLOAD_DIR / "data" / ".complete").exists():
        return DOWNLOAD_DIR / "data"
    return download_pklot(DOWNLOAD_DIR)

DATA_ROOT = find_data_root()
print("DATA_ROOT =", DATA_ROOT)

## 1. Đọc label

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2

import pandas as pd

CLASS_NAMES = {0: "vacant", 1: "occupied"}

def load_boxes(data_root: Path) -> pd.DataFrame:
    # Đọc toàn bộ label YOLO thành bảng, mỗi dòng là một ô đỗ (bbox theo pixel)
    rows = []
    for split in SPLITS:
        for txt in sorted((data_root / "labels" / split).glob("*.txt")):
            camera, date, time = txt.stem.split("_", 2)
            for idx, line in enumerate(txt.read_text().split("\n")):
                if not line.strip():
                    continue
                cls, xc, yc, w, h = line.split()
                xc, yc, w, h = float(xc) * IMG_W, float(yc) * IMG_H, float(w) * IMG_W, float(h) * IMG_H
                rows.append((split, txt.stem, camera, date, time, idx, int(cls),
                             xc - w / 2, yc - h / 2, w, h))
    return pd.DataFrame(rows, columns=["split", "frame", "camera", "date", "time", "box_idx", "cls",
                                       "x", "y", "w", "h"])

boxes = load_boxes(DATA_ROOT)
print(f"{boxes['frame'].nunique():,} ảnh, {len(boxes):,} ô đỗ")
boxes.head()

In [ ]:
# Kiểm tra kích thước ảnh thực tế khớp với IMG_W × IMG_H dùng để quy đổi bbox
for split in SPLITS:
    sample = next((DATA_ROOT / "images" / split).glob("*.jpg"))
    h, w = cv2.imread(str(sample)).shape[:2]
    assert (w, h) == (IMG_W, IMG_H), f"{sample.name}: {w}x{h}"
print("Kích thước ảnh OK:", IMG_W, "x", IMG_H)

## 2. Phân bố theo tập / camera / lớp

In [ ]:
frames = boxes.groupby(["split", "frame", "camera", "date", "time"], as_index=False).agg(
    n_boxes=("cls", "size"), occupied=("cls", "sum"))
frames["occupancy"] = frames["occupied"] / frames["n_boxes"]
frames["hour"] = frames["time"].str[:2].astype(int)

print("Số ảnh theo tập × camera:")
display(pd.crosstab(frames["split"], frames["camera"], margins=True).loc[SPLITS + ["All"]])
print("Số ô đỗ theo tập × lớp:")
display(pd.crosstab(boxes["split"], boxes["cls"].map(CLASS_NAMES), margins=True).loc[SPLITS + ["All"]])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
pd.crosstab(boxes["camera"], boxes["cls"].map(CLASS_NAMES)).plot.bar(ax=axes[0], rot=0, title="Số ô đỗ theo camera")
frames.boxplot(column="n_boxes", by="camera", ax=axes[1])
axes[1].set_title("Số ô gán nhãn / ảnh")
frames.groupby(["hour", "camera"])["occupancy"].mean().unstack().plot(ax=axes[2], marker="o",
                                                                      title="Tỷ lệ lấp đầy trung bình theo giờ")
axes[2].set_ylabel("occupied / tổng ô")
fig.suptitle("")
plt.tight_layout()
plt.savefig(WORK_DIR / "eda_distribution.png", dpi=120)
plt.show()

## 3. Kích thước bbox
Quyết định kích thước resize khi cắt crop (mặc định 128×128) và mức nới rộng box.

In [ ]:
print(boxes.groupby("camera")[["w", "h"]].describe().round(1).T)
fig, ax = plt.subplots(figsize=(6, 5))
for cam, g in boxes.groupby("camera"):
    s = g.sample(min(3000, len(g)), random_state=42)
    ax.scatter(s["w"], s["h"], s=4, alpha=0.3, label=cam)
ax.set_xlabel("width (px)"); ax.set_ylabel("height (px)"); ax.set_title("Kích thước bbox ô đỗ")
ax.legend(markerscale=4)
plt.savefig(WORK_DIR / "eda_box_sizes.png", dpi=120)
plt.show()

## 4. Kiểm tra rò rỉ giữa các tập

In [ ]:
groups = frames.groupby(["camera", "date"])["split"].nunique()
assert (groups == 1).all(), groups[groups > 1]
print("OK – mỗi (camera, ngày) chỉ thuộc một tập.")
print(frames.groupby("split")["date"].nunique().rename("số ngày").to_string())

## 5. Ảnh mẫu có bbox (xanh = vacant, đỏ = occupied)

In [ ]:
COLORS = {0: (0, 200, 0), 1: (0, 0, 255)}   # BGR

cameras = sorted(frames["camera"].unique())
fig, axes = plt.subplots(1, len(cameras), figsize=(6 * len(cameras), 4))
for ax, cam in zip(np.atleast_1d(axes), cameras):
    row = frames[(frames["camera"] == cam) & (frames["split"] == "train")].sample(1, random_state=0).iloc[0]
    img = cv2.imread(str(DATA_ROOT / "images" / row["split"] / f"{row['frame']}.jpg"))
    for b in boxes[boxes["frame"] == row["frame"]].itertuples():
        cv2.rectangle(img, (int(b.x), int(b.y)), (int(b.x + b.w), int(b.y + b.h)), COLORS[b.cls], 2)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{cam} – {row['date']} {row['time']} ({row['occupied']}/{row['n_boxes']} occupied)")
    ax.axis("off")
plt.tight_layout()
plt.savefig(WORK_DIR / "eda_samples.png", dpi=120)
plt.show()

## Ghi chú cho báo cáo
- Train/valid/test chia theo (camera, ngày) → đánh giá trên **ngày mới**, nhưng **cùng 3 camera**. Cần test thêm trên camera mới (CNRPark-EXT / video tự gán nhãn).
- Bbox là hình chữ nhật thẳng: với camera nhìn chéo (PUCPR) crop có thể dính xe ô bên cạnh – xem ảnh mẫu ở mục 5.
- Repo không có nhãn thời tiết theo ảnh; tập test chỉ có 104 ảnh mưa → không đánh giá riêng theo thời tiết.
- Lưu `eda_distribution.png`, `eda_box_sizes.png`, `eda_samples.png` để đưa vào chương Dữ liệu.